In [ ]:
import pandas as pd
import numpy  as np
import seaborn as sns


# Örnek bir veri seti oluşturma
np.random.seed(42)

data = {
    "ID": range(1, 101),  # 1'den 100'e kadar ID sütunu
    "Kategori": np.random.choice(["A", "B", "C", "D", "E"], size=100),  # Kategorik veri
    "Gelir": np.random.randint(2000, 10000, size=100),  # Gelir sütunu
    "Gider": np.random.randint(1000, 9000, size=100),  # Gider sütunu
    "Kar": np.random.randint(-2000, 5000, size=100),  # Kar sütunu (Pozitif ve negatif değerler içerebilir)
    "Memnuniyet": np.random.choice([1, 2, 3, 4, 5], size=100),  # 1-5 arasında müşteri memnuniyeti
}

# Eksik veri ekleme (bazı rastgele hücreleri NaN yapalım)
df_example = pd.DataFrame(data)
for col in ["Gelir", "Gider", "Kar"]:
    df_example.loc[df_example.sample(frac=0.1).index, col] = np.nan  # %10 eksik veri ekleme

# Dosyayı kaydetme
file_path = "example_dataset.csv"
df_example.to_csv(file_path, index=False)

# Kullanıcıya indirme bağlantısı sağlama
file_path


In [ ]:
hitters = pd.read_csv("example_dataset.csv")
df=hitters.copy()
missing_columns = df.columns[df.isnull().sum() > 0]

In [ ]:
df.info()

In [ ]:
df[df.isnull().any(axis=1)]

In [ ]:
df.isnull().sum()

In [ ]:
import missingno as msno
msno.bar(df)

In [ ]:
msno.matrix(df)

Bu bölümde ne yapılıyor?

df.copy() → Veri setinin bir kopyası oluşturuluyor (orijinali bozmamak için).
fillna("Unknown") → Eksik olan hücreleri "Unknown" değeri ile dolduruyor.<br>
Bu yöntem genellikle kategorik veriler için kullanılır (örneğin, eksik şehir veya kategori bilgileri).

In [ ]:
# 1️⃣ Global değişken (NULL, Unknown) ile doldurma
df_global = df.copy()
df_global[missing_columns] = df_global[missing_columns].fillna("Unknown")

# Sonuçları gösterme
print("Global Değişken ile Doldurulmuş Veri:")
display(df_global)


Bu bölümde ne yapılıyor?

Sadece sayısal sütunlarda eksik veri varsa, bunlar sütunun ortalama değeri ile dolduruluyor.<br>
dtype in ['float64', 'int64'] → Yalnızca sayısal veriler üzerinde işlem yapılıyor.<br>
Örnek:<br>
Eğer maaş sütununda eksik veri varsa, tüm maaşların ortalaması alınır ve eksik yerlere yazılır.

In [ ]:
# 2️⃣ Ortalama ile doldurma
df_mean = df.copy()
for col in missing_columns:
    if df_mean[col].dtype in ['float64', 'int64']:  
        df_mean[col] = df_mean[col].fillna(df_mean[col].mean())

print("Ortalama Değer ile Doldurulmuş Veri:")
display(df_mean)


Eğer veri setinde bir "Class" sütunu varsa, eksik değerleri o sınıfa ait ortalama ile doldurur.<br>
Örneğin: Bir sporcu grubunda, aynı takımda olan oyuncuların maaşları birbirine yakın olabilir. <br>
Eksik maaşları aynı takımdaki diğer oyuncuların ortalaması ile dolduruyoruz.<br>

In [ ]:
# 3️⃣ Aynı sınıfa ait kayıtların ortalaması ile doldurma (Eğer sınıf etiketi varsa)
df_class_mean = df.copy()
  
for col in missing_columns:
    if df_class_mean[col].dtype in ['float64', 'int64']:  
        df_class_mean[col] = df_class_mean.groupby("Kategori")[col].transform(lambda x: x.fillna(x.mean()))

print("Sınıf Ortalaması ile Doldurulmuş Veri:")
display(df_class_mean)


Eksik veriler, sütundaki en yaygın (mod) değerle dolduruluyor.<br>
Özellikle kategorik veriler için uygundur.

In [ ]:
# 4️⃣ En fazla tekrar eden (mod) ile doldurma
df_mode = df.copy()
for col in missing_columns:
    df_mode[col] = df_mode[col].fillna(df_mode[col].mode()[0])


print("En Fazla Olasılığa Sahip Değer ile Doldurulmuş Veri:")
display(df_mode)